In [2]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../datasets/master/master_final_2025.csv')

In [ ]:
def ols_manual(y, X, feature_names):
    """Manual OLS with intercept, returns coef table + R2, adj R2"""
    X = np.column_stack([np.ones(len(X)), X])
    names = ['const'] + feature_names
    n, k = X.shape
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    resid = y - y_hat
    sse = resid @ resid
    sst = ((y - y.mean())**2).sum()
    r2 = 1 - sse/sst
    adj_r2 = 1 - (1-r2)*(n-1)/(n-k)
    sigma2 = sse/(n-k)
    XtX_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    t_stat = beta/se
    p_val = 2*(1-stats.t.cdf(np.abs(t_stat), df=n-k))
    result = pd.DataFrame({'coef':beta, 'se':se, 't':t_stat, 'p_value':p_val}, index=names)
    return result, r2, adj_r2, n



In [4]:
print("MODEL 1: literacy_total ~ akses fasilitas + densitas sekolah + perpustakaan")
print("(n=38, semua provinsi)")
print("="*70)
feats = ['facility_ratio_shs','school_density_primary','library_per_1000pupils']
sub = df.dropna(subset=feats+['literacy_total'])
X = sub[feats].values
y = sub['literacy_total'].values
res, r2, adj_r2, n = ols_manual(y, X, feats)
print(res.round(4))
print(f"\nR2 = {r2:.3f}, Adj R2 = {adj_r2:.3f}, n = {n}")
print("\n"+"="*70)

MODEL 1: literacy_total ~ akses fasilitas + densitas sekolah + perpustakaan
(n=38, semua provinsi)
                           coef      se        t  p_value
const                   83.6308  4.5553  18.3589   0.0000
facility_ratio_shs      16.9051  6.9073   2.4474   0.0197
school_density_primary   1.2212  0.5000   2.4423   0.0199
library_per_1000pupils   0.1464  0.3964   0.3692   0.7143

R2 = 0.206, Adj R2 = 0.136, n = 38



In [5]:
print("MODEL 2: completion_senior_high ~ akses fasilitas + densitas sekolah + perpustakaan + gap_eys")
print("(n=34, exclude 4 provinsi pemekaran Papua tanpa data completion)")
print("="*70)
feats2 = ['facility_ratio_shs','school_density_primary','library_per_1000pupils','gap_eys']
sub2 = df.dropna(subset=feats2+['completion_senior_high'])
X2 = sub2[feats2].values
y2 = sub2['completion_senior_high'].values
res2, r2_2, adj_r2_2, n2 = ols_manual(y2, X2, feats2)
print(res2.round(4))
print(f"\nR2 = {r2_2:.3f}, Adj R2 = {adj_r2_2:.3f}, n = {n2}")

MODEL 2: completion_senior_high ~ akses fasilitas + densitas sekolah + perpustakaan + gap_eys
(n=34, exclude 4 provinsi pemekaran Papua tanpa data completion)
                           coef       se       t  p_value
const                   66.1404  12.0516  5.4881   0.0000
facility_ratio_shs      29.6291  16.1544  1.8341   0.0769
school_density_primary  -1.1861   1.1535 -1.0283   0.3123
library_per_1000pupils   0.9880   0.7585  1.3025   0.2030
gap_eys                 -4.6071   6.5587 -0.7024   0.4880

R2 = 0.327, Adj R2 = 0.234, n = 34


In [6]:
print("MODEL 3: completion_senior_high ~ akses fasilitas + dummy region (kontrol wilayah)")
print("="*70)
sub3 = df.dropna(subset=['completion_senior_high','facility_ratio_shs','school_density_primary'])
region_dummies = pd.get_dummies(sub3['region'], drop_first=True, dtype=float)  # baseline = Bali-Nusra (alphabetically first dropped)
feats3 = ['facility_ratio_shs','school_density_primary'] + list(region_dummies.columns)
X3 = pd.concat([sub3[['facility_ratio_shs','school_density_primary']], region_dummies], axis=1).values
y3 = sub3['completion_senior_high'].values
res3, r2_3, adj_r2_3, n3 = ols_manual(y3, X3, feats3)
print(res3.round(4))
print(f"\nR2 = {r2_3:.3f}, Adj R2 = {adj_r2_3:.3f}, n = {n3}")
print("(baseline region = Bali-Nusra; koefisien region menunjukkan selisih completion SMA vs Bali-Nusra, mengontrol akses fasilitas)")

MODEL 3: completion_senior_high ~ akses fasilitas + dummy region (kontrol wilayah)
                           coef       se       t  p_value
const                   47.6640  15.2172  3.1322   0.0043
facility_ratio_shs      36.9696  18.1892  2.0325   0.0524
school_density_primary   0.3165   1.4965  0.2115   0.8341
Jawa                     8.8236   7.0966  1.2434   0.2248
Kalimantan               6.6684   7.1997  0.9262   0.3629
Maluku-Papua             1.8532   7.6328  0.2428   0.8101
Sulawesi                 1.6849   7.1505  0.2356   0.8156
Sumatera                 9.8180   6.5735  1.4936   0.1473

R2 = 0.365, Adj R2 = 0.194, n = 34
(baseline region = Bali-Nusra; koefisien region menunjukkan selisih completion SMA vs Bali-Nusra, mengontrol akses fasilitas)
